In [1]:
# ============================================================
# HENRY HUB NATURAL GAS SPOT PRICE
# K-MEANS CLUSTERING + NLP / TF-IDF ANALYSIS
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================

# Run this cell if the packages are not already installed.
# Uncomment the following line if needed:

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords
nltk.download('stopwords')

# Load English stopwords
stop_words = set(stopwords.words('english'))

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

sns.set_style("whitegrid")

print("Libraries imported successfully.")


# ============================================================
# 3. LOAD DATASET
# ============================================================

file_path = "Henry_Hub_Natural_Gas_Spot_Price.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 4. DATASET INFORMATION
# ============================================================

print("Dataset Information:")
print("=" * 60)

df.info()

print("\nMissing Values:")
print("=" * 60)

print(df.isnull().sum())


# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nCleaned column names:")
print(df.columns.tolist())


# ============================================================
# 6. IDENTIFY DATE COLUMN
# ============================================================

date_candidates = [
    col for col in df.columns
    if (
        'date' in col
        or 'period' in col
        or 'month' in col
    )
]

print("\nPossible date columns:")
print(date_candidates)

if len(date_candidates) == 0:
    raise ValueError(
        "No date column was detected. "
        "Please check the CSV column names."
    )

date_col = date_candidates[0]

print("\nSelected date column:", date_col)


# ============================================================
# 7. IDENTIFY PRICE COLUMN
# ============================================================

price_candidates = [
    col for col in df.columns
    if (
        'price' in col
        or 'spot' in col
        or 'value' in col
    )
]

print("\nPossible price columns:")
print(price_candidates)

if len(price_candidates) == 0:
    raise ValueError(
        "No price column was detected. "
        "Please check the CSV column names."
    )

price_col = price_candidates[0]

print("\nSelected price column:", price_col)


# ============================================================
# 8. CONVERT DATE COLUMN
# ============================================================

df[date_col] = pd.to_datetime(
    df[date_col],
    errors='coerce'
)

print("\nDate conversion completed.")

print(
    "Invalid dates:",
    df[date_col].isna().sum()
)


# ============================================================
# 9. CLEAN PRICE COLUMN
# ============================================================

df[price_col] = (
    df[price_col]
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df[price_col] = pd.to_numeric(
    df[price_col],
    errors='coerce'
)

print("\nPrice conversion completed.")

print(
    "Invalid prices:",
    df[price_col].isna().sum()
)


# ============================================================
# 10. REMOVE MISSING VALUES
# ============================================================

print("\nDataset before cleaning:", df.shape)

df = df.dropna(
    subset=[date_col, price_col]
).copy()

print("Dataset after cleaning:", df.shape)


# ============================================================
# 11. SORT BY DATE
# ============================================================

df = df.sort_values(
    by=date_col
).reset_index(drop=True)

print("\nDataset sorted by date.")

display(df.head())


# ============================================================
# 12. DESCRIPTIVE STATISTICS
# ============================================================

print("\nDescriptive Statistics:")
print("=" * 60)

display(
    df[price_col].describe()
)


# ============================================================
# 13. PRICE OVER TIME
# ============================================================

plt.figure(figsize=(15, 7))

plt.plot(
    df[date_col],
    df[price_col],
    linewidth=1.5
)

plt.title(
    "Henry Hub Natural Gas Spot Price Over Time",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Natural Gas Price")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
# 14. HISTOGRAM OF NATURAL GAS PRICES
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df[price_col],
    bins=30
)

plt.title(
    "Distribution of Henry Hub Natural Gas Prices",
    fontsize=16
)

plt.xlabel("Natural Gas Price")
plt.ylabel("Frequency")

plt.tight_layout()

plt.show()


# ============================================================
# 15. BOX PLOT
# ============================================================

plt.figure(figsize=(10, 5))

sns.boxplot(
    x=df[price_col]
)

plt.title(
    "Box Plot of Natural Gas Prices",
    fontsize=16
)

plt.xlabel("Natural Gas Price")

plt.tight_layout()

plt.show()


# ============================================================
# 16. FEATURE ENGINEERING
# ============================================================

df['year'] = df[date_col].dt.year
df['month'] = df[date_col].dt.month
df['quarter'] = df[date_col].dt.quarter

# Price difference
df['price_change'] = (
    df[price_col].diff()
)

# Percentage change
df['price_pct_change'] = (
    df[price_col].pct_change() * 100
)

# 3-period moving average
df['rolling_mean_3'] = (
    df[price_col]
    .rolling(window=3)
    .mean()
)

# 3-period rolling volatility
df['rolling_std_3'] = (
    df[price_col]
    .rolling(window=3)
    .std()
)

print("Feature engineering completed.")

display(df.head(10))


# ============================================================
# 17. REMOVE NaN VALUES CREATED BY FEATURE ENGINEERING
# ============================================================

df_ml = df.dropna().copy()

print(
    "Machine learning dataset shape:",
    df_ml.shape
)


# ============================================================
# 18. SELECT FEATURES FOR K-MEANS
# ============================================================

features = [
    price_col,
    'month',
    'quarter',
    'price_change',
    'price_pct_change',
    'rolling_mean_3',
    'rolling_std_3'
]

print("\nFeatures selected for K-Means:")
for feature in features:
    print("-", feature)

X = df_ml[features].copy()


# ============================================================
# 19. STANDARDIZE FEATURES
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("\nFeature standardization completed.")

print(
    "Scaled data shape:",
    X_scaled.shape
)


# ============================================================
# 20. ELBOW METHOD
# ============================================================

inertia = []

K_range = range(2, 11)

for k in K_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(
        kmeans.inertia_
    )


# Plot elbow curve

plt.figure(figsize=(10, 6))

plt.plot(
    list(K_range),
    inertia,
    marker='o',
    linewidth=2
)

plt.title(
    "Elbow Method for K-Means",
    fontsize=16
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")

plt.xticks(list(K_range))

plt.grid(True)

plt.tight_layout()

plt.show()


# ============================================================
# 21. SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

for k in K_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(
        X_scaled
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_scores.append(score)


# Create results table

silhouette_results = pd.DataFrame({
    'Number_of_Clusters': list(K_range),
    'Silhouette_Score': silhouette_scores
})

print("\nSilhouette Scores:")
display(silhouette_results)


# ============================================================
# 22. PLOT SILHOUETTE SCORES
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    silhouette_results['Number_of_Clusters'],
    silhouette_results['Silhouette_Score'],
    marker='o',
    linewidth=2
)

plt.title(
    "Silhouette Score by Number of Clusters",
    fontsize=16
)

plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")

plt.grid(True)

plt.tight_layout()

plt.show()


# ============================================================
# 23. SELECT BEST K
# ============================================================

best_k = int(
    silhouette_results.loc[
        silhouette_results['Silhouette_Score'].idxmax(),
        'Number_of_Clusters'
    ]
)

best_score = silhouette_results.loc[
    silhouette_results['Silhouette_Score'].idxmax(),
    'Silhouette_Score'
]

print("Best K:", best_k)
print("Best Silhouette Score:", round(best_score, 4))


# ============================================================
# 24. FINAL K-MEANS MODEL
# ============================================================

kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df_ml['cluster'] = (
    kmeans_final.fit_predict(X_scaled)
)

print("\nK-Means clustering completed.")

print("\nNumber of observations per cluster:")

display(
    df_ml['cluster']
    .value_counts()
    .sort_index()
)


# ============================================================
# 25. CLUSTER SUMMARY
# ============================================================

cluster_summary = df_ml.groupby(
    'cluster'
)[price_col].agg([
    'count',
    'mean',
    'median',
    'min',
    'max',
    'std'
]).round(3)

print("\nCluster Summary:")
display(cluster_summary)


# ============================================================
# 26. CLUSTER CHARACTERISTICS
# ============================================================

cluster_characteristics = df_ml.groupby(
    'cluster'
).agg({
    price_col: 'mean',
    'price_change': 'mean',
    'price_pct_change': 'mean',
    'rolling_mean_3': 'mean',
    'rolling_std_3': 'mean'
}).round(3)

print("\nCluster Characteristics:")
display(cluster_characteristics)


# ============================================================
# 27. VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(15, 7))

sns.scatterplot(
    data=df_ml,
    x=date_col,
    y=price_col,
    hue='cluster',
    palette='tab10',
    s=60
)

plt.title(
    "Henry Hub Natural Gas Price K-Means Clusters",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Natural Gas Price")

plt.xticks(rotation=45)

plt.legend(
    title="Cluster"
)

plt.tight_layout()

plt.show()


# ============================================================
# 28. CLUSTER PRICE DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df_ml,
    x='cluster',
    y=price_col
)

plt.title(
    "Natural Gas Price Distribution by Cluster",
    fontsize=16
)

plt.xlabel("Cluster")
plt.ylabel("Natural Gas Price")

plt.tight_layout()

plt.show()


# ============================================================
# 29. CREATE MARKET TEXT FOR NLP
# ============================================================

# Calculate price thresholds

low_threshold = df_ml[price_col].quantile(
    0.33
)

high_threshold = df_ml[price_col].quantile(
    0.66
)


def get_price_level(price):

    if price <= low_threshold:
        return "low price"

    elif price <= high_threshold:
        return "medium price"

    else:
        return "high price"


def get_price_movement(change):

    if change < -5:
        return "strong decrease"

    elif change < 0:
        return "decrease"

    elif change <= 5:
        return "stable increase"

    else:
        return "strong increase"


def create_market_text(row):

    price_level = get_price_level(
        row[price_col]
    )

    movement = get_price_movement(
        row['price_pct_change']
    )

    return (
        f"natural gas "
        f"{price_level} "
        f"year {int(row['year'])} "
        f"month {int(row['month'])} "
        f"quarter {int(row['quarter'])} "
        f"{movement} "
        f"market cluster {int(row['cluster'])}"
    )


df_ml['market_text'] = df_ml.apply(
    create_market_text,
    axis=1
)

print("\nGenerated NLP text:")

display(
    df_ml[
        [
            date_col,
            price_col,
            'cluster',
            'market_text'
        ]
    ].head(10)
)


# ============================================================
# 30. NLP TEXT CLEANING
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove numbers and punctuation
    text = re.sub(
        r'[^a-zA-Z\s]',
        '',
        text
    )

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    return ' '.join(words)


df_ml['clean_text'] = (
    df_ml['market_text']
    .apply(clean_text)
)

print("\nCleaned NLP text:")

display(
    df_ml[
        ['market_text', 'clean_text']
    ].head(10)
)


# ============================================================
# 31. TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    df_ml['clean_text']
)

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)


# ============================================================
# 32. TF-IDF FEATURE NAMES
# ============================================================

tfidf_features = (
    vectorizer
    .get_feature_names_out()
)

print("\nTF-IDF Terms:")
print(tfidf_features)


# ============================================================
# 33. CALCULATE AVERAGE TF-IDF SCORE
# ============================================================

tfidf_scores = np.asarray(
    tfidf_matrix.mean(axis=0)
).flatten()

tfidf_df = pd.DataFrame({
    'term': tfidf_features,
    'tfidf_score': tfidf_scores
})

tfidf_df = tfidf_df.sort_values(
    'tfidf_score',
    ascending=False
)

print("\nTop TF-IDF Terms:")

display(
    tfidf_df.head(20)
)


# ============================================================
# 34. VISUALIZE TOP NLP TERMS
# ============================================================

top_terms = tfidf_df.head(20)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_terms,
    x='tfidf_score',
    y='term'
)

plt.title(
    "Top 20 TF-IDF Terms",
    fontsize=16
)

plt.xlabel("Average TF-IDF Score")
plt.ylabel("Term")

plt.tight_layout()

plt.show()


# ============================================================
# 35. WORD CLOUD
# ============================================================

all_text = ' '.join(
    df_ml['clean_text']
)

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    max_words=100
).generate(all_text)

plt.figure(figsize=(15, 8))

plt.imshow(
    wordcloud,
    interpolation='bilinear'
)

plt.axis('off')

plt.title(
    "Henry Hub Natural Gas NLP Word Cloud",
    fontsize=18
)

plt.show()


# ============================================================
# 36. K-MEANS ON TF-IDF NLP FEATURES
# ============================================================

nlp_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df_ml['nlp_cluster'] = (
    nlp_kmeans.fit_predict(
        tfidf_matrix
    )
)

print("\nNLP K-Means clustering completed.")

print("\nNLP cluster counts:")

display(
    df_ml['nlp_cluster']
    .value_counts()
    .sort_index()
)


# ============================================================
# 37. NLP CLUSTER SILHOUETTE SCORE
# ============================================================

nlp_silhouette = silhouette_score(
    tfidf_matrix,
    df_ml['nlp_cluster']
)

print(
    "NLP K-Means Silhouette Score:",
    round(nlp_silhouette, 4)
)


# ============================================================
# 38. COMPARE NUMERIC AND NLP CLUSTERS
# ============================================================

cluster_comparison = pd.crosstab(
    df_ml['cluster'],
    df_ml['nlp_cluster']
)

print(
    "\nComparison between Numeric K-Means "
    "and NLP K-Means:"
)

display(cluster_comparison)


# ============================================================
# 39. NLP CLUSTER SUMMARY
# ============================================================

nlp_cluster_summary = df_ml.groupby(
    'nlp_cluster'
)[price_col].agg([
    'count',
    'mean',
    'median',
    'min',
    'max',
    'std'
]).round(3)

print("\nNLP Cluster Price Summary:")

display(nlp_cluster_summary)


# ============================================================
# 40. VISUALIZE NLP CLUSTERS OVER TIME
# ============================================================

plt.figure(figsize=(15, 7))

sns.scatterplot(
    data=df_ml,
    x=date_col,
    y=price_col,
    hue='nlp_cluster',
    palette='tab10',
    s=60
)

plt.title(
    "Henry Hub Natural Gas Price - NLP K-Means Clusters",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Natural Gas Price")

plt.xticks(rotation=45)

plt.legend(
    title="NLP Cluster"
)

plt.tight_layout()

plt.show()


# ============================================================
# 41. FIND HIGHEST AND LOWEST PRICE CLUSTERS
# ============================================================

cluster_means = (
    df_ml
    .groupby('cluster')[price_col]
    .mean()
    .sort_values()
)

lowest_cluster = cluster_means.index[0]
highest_cluster = cluster_means.index[-1]

print(
    "Lowest-price cluster:",
    lowest_cluster
)

print(
    "Average price:",
    round(
        cluster_means.iloc[0],
        3
    )
)

print(
    "\nHighest-price cluster:",
    highest_cluster
)

print(
    "Average price:",
    round(
        cluster_means.iloc[-1],
        3
    )
)


# ============================================================
# 42. FIND DATES BELONGING TO EACH CLUSTER
# ============================================================

for cluster in sorted(
    df_ml['cluster'].unique()
):

    cluster_data = df_ml[
        df_ml['cluster'] == cluster
    ]

    print("\n")
    print("=" * 60)
    print(f"CLUSTER {cluster}")
    print("=" * 60)

    print(
        "Average price:",
        round(
            cluster_data[price_col].mean(),
            3
        )
    )

    print(
        "Minimum price:",
        round(
            cluster_data[price_col].min(),
            3
        )
    )

    print(
        "Maximum price:",
        round(
            cluster_data[price_col].max(),
            3
        )
    )

    print(
        "Number of observations:",
        len(cluster_data)
    )


# ============================================================
# 43. SAVE CLEANED DATA
# ============================================================

cleaned_file = (
    "Henry_Hub_Natural_Gas_Cleaned.csv"
)

df_ml.to_csv(
    cleaned_file,
    index=False
)

print(
    f"\nCleaned dataset saved as: {cleaned_file}"
)


# ============================================================
# 44. SAVE CLUSTER SUMMARY
# ============================================================

cluster_summary_file = (
    "Henry_Hub_KMeans_Cluster_Summary.csv"
)

cluster_summary.to_csv(
    cluster_summary_file
)

print(
    f"Cluster summary saved as: "
    f"{cluster_summary_file}"
)


# ============================================================
# 45. SAVE TF-IDF RESULTS
# ============================================================

tfidf_file = (
    "Henry_Hub_NLP_TFIDF_Terms.csv"
)

tfidf_df.to_csv(
    tfidf_file,
    index=False
)

print(
    f"TF-IDF results saved as: {tfidf_file}"
)


# ============================================================
# 46. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL ANALYSIS RESULTS")
print("=" * 70)

print(
    f"Original dataset rows: {len(df)}"
)

print(
    f"Machine learning rows: {len(df_ml)}"
)

print(
    f"Optimal number of clusters: {best_k}"
)

print(
    f"Best numerical K-Means silhouette score: "
    f"{best_score:.4f}"
)

print(
    f"NLP K-Means silhouette score: "
    f"{nlp_silhouette:.4f}"
)

print(
    f"Lowest-price cluster: {lowest_cluster}"
)

print(
    f"Highest-price cluster: {highest_cluster}"
)

print("\nAnalysis completed successfully.")

Libraries imported successfully.
Dataset loaded successfully.
Dataset shape: (5917, 2)

Column names:
['Day', 'Henry Hub Natural Gas Spot Price Dollars per Million Btu']

First 5 rows:


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Day,Henry Hub Natural Gas Spot Price Dollars per Million Btu
0,07/20/2020,1.71
1,07/17/2020,1.79
2,07/16/2020,1.79
3,07/15/2020,1.76
4,07/14/2020,1.74


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 5917 entries, 0 to 5916
Data columns (total 2 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   Day                                                       5917 non-null   str    
 1   Henry Hub Natural Gas Spot Price Dollars per Million Btu  5916 non-null   float64
dtypes: float64(1), str(1)
memory usage: 148.7 KB

Missing Values:
Day                                                         0
Henry Hub Natural Gas Spot Price Dollars per Million Btu    1
dtype: int64

Cleaned column names:
['day', 'henry_hub_natural_gas_spot_price_dollars_per_million_btu']

Possible date columns:
[]


ValueError: No date column was detected. Please check the CSV column names.